In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')

In [3]:
test_df         = pd.read_parquet("../data/processed/test_split.parquet")
model_corrected = joblib.load("../artifacts/models/model_corrected.pkl")
feature_cols    = joblib.load("../artifacts/models/feature_cols.pkl")

In [4]:
X_test = test_df[feature_cols]

test_df['predicted_demand'] = model_corrected.predict(X_test)

print(test_df[['store_id', 'product_id', 'dt', 
               'sale_amount', 'predicted_demand']].head(10))

   store_id  product_id         dt  sale_amount  predicted_demand
0         0           4 2024-06-17          0.1          0.720411
1         0           4 2024-06-18          0.9          0.715400
2         0           4 2024-06-19          0.0          0.952025
3         0           4 2024-06-20          1.2          0.699890
4         0           4 2024-06-21          0.3          1.051217
5         0           4 2024-06-22          2.1          1.965793
6         0           4 2024-06-23          1.6          2.181839
7         0           4 2024-06-24          0.0          0.851630
8         0           4 2024-06-25          0.3          0.454549
9         0           6 2024-06-17          1.2          1.460190


In [5]:
def recommend_inventory(
    forecast_demand,       # array of daily predicted demand
    current_inventory,     # current stock on hand
    lead_time_days=3,      # days to receive new stock
    service_level=0.95,    # how often we want to avoid stockout
    holding_cost=2.0,      # cost per unit per day in storage
    stockout_cost=20.0     # cost per unit of lost sale
):
    avg_daily_demand = np.mean(forecast_demand)
    std_daily_demand = np.std(forecast_demand)
    
    # Service level factor (z-score)
    from scipy.stats import norm
    z = norm.ppf(service_level)
    
    # Safety stock covers demand variability during lead time
    safety_stock = z * std_daily_demand * np.sqrt(lead_time_days)
    
    # Expected demand during lead time
    demand_during_lead_time = avg_daily_demand * lead_time_days
    
    # Reorder point = demand during lead time + safety stock
    reorder_point = demand_during_lead_time + safety_stock
    
    # How much to order
    reorder_qty = max(0, reorder_point - current_inventory)
    
    # Risk and cost estimates
    stockout_risk   = 1 - service_level
    overstock_units = max(0, current_inventory - reorder_point)
    estimated_cost  = (overstock_units * holding_cost) + (reorder_qty * holding_cost)
    
    return {
        'avg_daily_demand':        round(avg_daily_demand, 3),
        'safety_stock':            round(safety_stock, 3),
        'reorder_point':           round(reorder_point, 3),
        'recommended_reorder_qty': round(reorder_qty, 3),
        'expected_stockout_risk':  round(stockout_risk, 3),
        'overstock_units':         round(overstock_units, 3),
        'estimated_cost':          round(estimated_cost, 3)
    }

In [7]:
# Pick store 0, product 4 forecast
mask = (test_df['store_id'] == 0) & (test_df['product_id'] == 4)
forecast = test_df[mask]['predicted_demand'].values

In [8]:

print(f"Forecast for store 0, product 4 ({len(forecast)} days):")
print(forecast)

Forecast for store 0, product 4 (9 days):
[0.7204106  0.71539986 0.9520247  0.6998899  1.0512174  1.9657935
 2.1818385  0.8516296  0.4545492 ]


In [9]:
result = recommend_inventory(
    forecast_demand=forecast,
    current_inventory=10,
    lead_time_days=3,
    service_level=0.95,
    holding_cost=2.0,
    stockout_cost=20.0
)

print("\nInventory Recommendation:")
for k, v in result.items():
    print(f"  {k}: {v}")


Inventory Recommendation:
  avg_daily_demand: 1.065999984741211
  safety_stock: 1.607
  reorder_point: 4.805
  recommended_reorder_qty: 0
  expected_stockout_risk: 0.05
  overstock_units: 5.195
  estimated_cost: 10.39


In [10]:
result_low = recommend_inventory(
    forecast_demand=forecast,
    current_inventory=2,
    lead_time_days=3,
    service_level=0.95,
    holding_cost=2.0,
    stockout_cost=20.0
)

print("Inventory Recommendation (low stock scenario):")
for k, v in result_low.items():
    print(f"  {k}: {v}")

Inventory Recommendation (low stock scenario):
  avg_daily_demand: 1.065999984741211
  safety_stock: 1.607
  reorder_point: 4.805
  recommended_reorder_qty: 2.805
  expected_stockout_risk: 0.05
  overstock_units: 0
  estimated_cost: 5.61
